# Model B 통합 전처리 노트북

제주 SMP 가격 및 마이너스 SMP 구간 예측 모델(Model B)을 위한 최종 학습 데이터셋을
원천 CSV에서 한 번에 생성한다.

실행 순서:
1. SMP feature
2. Calendar feature
3. Weather feature
4. Renewable actual
5. Renewable forecast
6. Fuel cost feature
7. Oil / USD feature
8. Final merge: `model_b_data.csv`

기준 시간축은 `2024-06-01 00:00:00`부터 `2026-03-31 23:00:00`까지 16056시간이다.



In [23]:
from pathlib import Path
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 300)

# Colab에서는 Google Drive를 mount한다. 로컬에서 실행할 때는 이 블록을 건너뛴다.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("google.colab 환경이 아니므로 Drive mount를 건너뜁니다.")

DRIVE_BASE_DIR = Path("/content/drive/MyDrive/Projects/2026_AX_ECOTHON/ax_team/ax_data/model_b")

if DRIVE_BASE_DIR.exists():
    BASE_DIR = DRIVE_BASE_DIR
    RAW_DIR = BASE_DIR / "RAW"
    PREP_DIR = BASE_DIR / "전처리"
else:
    # 로컬 검수용 fallback. 프로젝트 루트에서 실행하면 ./model_b_data를 RAW로 사용한다.
    BASE_DIR = Path.cwd()
    RAW_DIR = BASE_DIR / "model_b_data"
    PREP_DIR = RAW_DIR / "전처리"

PREP_DIR.mkdir(parents=True, exist_ok=True)

FINAL_START = pd.Timestamp("2024-06-01 00:00:00")
FINAL_END = pd.Timestamp("2026-03-31 23:00:00")
FINAL_INDEX = pd.date_range(FINAL_START, FINAL_END, freq="h")
EXPECTED_ROWS = len(FINAL_INDEX)

assert EXPECTED_ROWS == 16056

SPECIAL_WINDOW_DAYS = 3
RANDOM_STATE = 42
MIN_TRAIN_DAYS = 60
VALID_DAYS = 30
BASELINE_MONTH = pd.Timestamp("2024-06-01")

print("IN_COLAB:", IN_COLAB)
print("BASE_DIR:", BASE_DIR)
print("RAW_DIR :", RAW_DIR)
print("PREP_DIR:", PREP_DIR)
print("FINAL_START:", FINAL_START)
print("FINAL_END  :", FINAL_END)
print("EXPECTED_ROWS:", EXPECTED_ROWS)

if not RAW_DIR.exists():
    raise FileNotFoundError(f"RAW_DIR가 존재하지 않습니다: {RAW_DIR}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB: True
BASE_DIR: /content/drive/MyDrive/Projects/2026_AX_ECOTHON/ax_team/ax_data/model_b
RAW_DIR : /content/drive/MyDrive/Projects/2026_AX_ECOTHON/ax_team/ax_data/model_b/RAW
PREP_DIR: /content/drive/MyDrive/Projects/2026_AX_ECOTHON/ax_team/ax_data/model_b/전처리
FINAL_START: 2024-06-01 00:00:00
FINAL_END  : 2026-03-31 23:00:00
EXPECTED_ROWS: 16056


## 공통 함수

여러 원천 파일에서 반복적으로 필요한 인코딩 처리, 한글 파일명 정규화,
timestamp 변환, 최종 시간축 검수를 한 곳에 모았다.



In [24]:
ENCODINGS = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]

MONTH_MAP = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}


def nfc(text):
    return unicodedata.normalize("NFC", str(text))


def resolve_raw_file(expected_name):
    target = nfc(expected_name)
    matches = [p for p in RAW_DIR.glob("*.csv") if nfc(p.name) == target]

    if not matches:
        available = sorted(nfc(p.name) for p in RAW_DIR.glob("*.csv"))
        raise FileNotFoundError(
            f"RAW 파일을 찾을 수 없습니다: {expected_name}\n"
            f"현재 RAW_DIR CSV 목록: {available}"
        )
    if len(matches) > 1:
        raise RuntimeError(f"동일 이름으로 정규화되는 파일이 여러 개입니다: {matches}")

    return matches[0]


def read_csv_auto(path, **kwargs):
    last_error = None

    for enc in ENCODINGS:
        try:
            df = pd.read_csv(path, encoding=enc, **kwargs)
            df.attrs["source_encoding"] = enc
            return df
        except UnicodeDecodeError as e:
            last_error = e

    raise last_error


def clean_columns(df):
    out = df.copy()
    out.columns = [str(c).replace("\ufeff", "").strip() for c in out.columns]
    return out


def to_numeric_clean(s):
    out = (
        s.astype(str)
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace("−", "-", regex=False)
    )
    out = out.replace({"": np.nan, "-": np.nan, "nan": np.nan, "None": np.nan})
    return pd.to_numeric(out, errors="coerce")


def parse_date_clean(values):
    s = pd.Series(values).astype(str).str.strip()
    s = s.str.replace(".", "-", regex=False).str.replace("/", "-", regex=False)

    dt = pd.to_datetime(s, errors="coerce")
    if dt.isna().any():
        raise ValueError(f"날짜 파싱 실패 샘플: {s.loc[dt.isna()].head(10).tolist()}")

    return dt.dt.normalize()


def parse_hour_number(values):
    s = pd.Series(values).astype(str).str.strip()
    hour = s.str.extract(r"(\d{1,2})")[0]
    return pd.to_numeric(hour, errors="coerce").astype("Int64")


def make_timestamp_hour_ending(date_values, hour_values):
    """
    KPX/EPSIS의 1~24시 또는 01~24시 데이터를 hour-ending timestamp로 변환한다.

    - 01시 / 거래시간 1  -> 당일 01:00
    - 24시 / 거래시간 24 -> 다음 날 00:00
    """
    dates = parse_date_clean(date_values)
    hours = parse_hour_number(hour_values)

    if hours.isna().any():
        bad = pd.Series(hour_values).loc[hours.isna()].head(10).tolist()
        raise ValueError(f"시간 파싱 실패 샘플: {bad}")

    invalid = ~hours.between(1, 24)
    if invalid.any():
        bad = pd.Series(hour_values).loc[invalid].head(10).tolist()
        raise ValueError(f"1~24 범위 밖 시간 샘플: {bad}")

    return dates + pd.to_timedelta(hours.astype(int), unit="h")


def make_timestamp_from_date_time(date_values, time_values):
    ts = pd.to_datetime(
        pd.Series(date_values).astype(str).str.strip()
        + " "
        + pd.Series(time_values).astype(str).str.strip(),
        errors="coerce",
    )

    if ts.isna().any():
        bad = pd.DataFrame({
            "date": pd.Series(date_values).loc[ts.isna()],
            "time": pd.Series(time_values).loc[ts.isna()],
        }).head(10)
        display(bad)
        raise ValueError("날짜+시간 timestamp 파싱 실패가 있습니다.")

    return ts


def find_hour_columns(columns):
    hour_cols = []

    for col in columns:
        text = str(col).strip()
        match = re.fullmatch(r"(\d{1,2})\s*시", text)
        if match:
            hour = int(match.group(1))
            if 1 <= hour <= 24:
                hour_cols.append((col, hour))

    return [col for col, _ in sorted(hour_cols, key=lambda x: x[1])]


def wide_smp_to_long(df, value_name):
    df = clean_columns(df)

    if "기간" not in df.columns:
        raise KeyError("SMP 원천에 '기간' 컬럼이 없습니다.")

    hour_cols = find_hour_columns(df.columns)
    if len(hour_cols) != 24:
        raise ValueError(f"01~24시 컬럼이 24개가 아닙니다. 발견 {len(hour_cols)}개: {hour_cols}")

    long_df = df[["기간"] + hour_cols].melt(
        id_vars=["기간"],
        value_vars=hour_cols,
        var_name="hour_raw",
        value_name=value_name,
    )

    long_df["timestamp"] = make_timestamp_hour_ending(long_df["기간"], long_df["hour_raw"])
    long_df[value_name] = to_numeric_clean(long_df[value_name])

    return (
        long_df[["timestamp", value_name]]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )


def parse_korean_2digit_date(values):
    s = pd.Series(values).astype(str).str.strip()
    compact = s.str.replace(r"\s+", "", regex=True)

    dt = pd.to_datetime(compact, format="%y년%m월%d일", errors="coerce")
    if dt.isna().any():
        bad = s.loc[dt.isna()].head(10).tolist()
        raise ValueError(f"한국어 날짜 파싱 실패 샘플: {bad}")

    return dt.dt.normalize()


def parse_epsis_month(value):
    text = str(value).strip()
    match = re.fullmatch(r"([A-Za-z]{3})\.(\d{2})", text)

    if not match:
        return pd.NaT

    month_text = match.group(1)
    year_2d = int(match.group(2))

    if month_text not in MONTH_MAP:
        return pd.NaT

    return pd.Timestamp(2000 + year_2d, MONTH_MAP[month_text], 1)


def find_fuel_column(raw_df, group_name, fuel_name):
    matches = []

    for col in raw_df.columns:
        col_group = str(col).split(".")[0]
        first_row_value = str(raw_df.loc[0, col]).strip()

        if col_group == group_name and first_row_value == fuel_name:
            matches.append(col)

    if len(matches) != 1:
        raise ValueError(
            f"{group_name}/{fuel_name} 컬럼을 정확히 1개 찾지 못했습니다. matches={matches}"
        )

    return matches[0]


def audit_time_index(df, name="df", show_nulls=True):
    temp = df.copy()
    temp["timestamp"] = pd.to_datetime(temp["timestamp"])

    unique_ts = pd.DatetimeIndex(temp["timestamp"].dropna().unique()).sort_values()
    in_final_ts = unique_ts[(unique_ts >= FINAL_START) & (unique_ts <= FINAL_END)]

    missing_ts = FINAL_INDEX.difference(in_final_ts)
    extra_ts = unique_ts.difference(FINAL_INDEX)

    summary = pd.DataFrame([{
        "name": name,
        "rows": len(temp),
        "columns": temp.shape[1],
        "min_timestamp": temp["timestamp"].min(),
        "max_timestamp": temp["timestamp"].max(),
        "duplicate_timestamp_rows": int(temp["timestamp"].duplicated().sum()),
        "expected_rows": EXPECTED_ROWS,
        "unique_timestamps_in_final": len(in_final_ts),
        "missing_timestamp_count": len(missing_ts),
        "extra_timestamp_count": len(extra_ts),
    }])

    display(summary)

    nulls = temp.isna().sum().sort_values(ascending=False)
    nulls = nulls[nulls > 0].to_frame("missing_count")

    if show_nulls:
        if len(nulls) > 0:
            display(nulls)
        else:
            print("결측 없음")

    if len(missing_ts) > 0:
        print("누락 timestamp 앞부분:", list(missing_ts[:10]))
        print("누락 timestamp 뒷부분:", list(missing_ts[-10:]))

    if len(extra_ts) > 0:
        print("기간 외 timestamp 앞부분:", list(extra_ts[:10]))
        print("기간 외 timestamp 뒷부분:", list(extra_ts[-10:]))

    return {
        "summary": summary.iloc[0].to_dict(),
        "missing_timestamps": missing_ts,
        "extra_timestamps": extra_ts,
        "null_counts": nulls,
    }


def align_to_final_index(df):
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"])

    out = out[
        (out["timestamp"] >= FINAL_START)
        & (out["timestamp"] <= FINAL_END)
    ].copy()

    dup_count = out["timestamp"].duplicated().sum()
    if dup_count > 0:
        dup_sample = out.loc[out["timestamp"].duplicated(), "timestamp"].head(10).tolist()
        raise ValueError(f"timestamp 중복이 있습니다. 중복 수={dup_count}, 샘플={dup_sample}")

    base = pd.DataFrame({"timestamp": FINAL_INDEX})
    return base.merge(out, on="timestamp", how="left")


def assert_final_timestamp(df, name):
    if df.shape[0] != EXPECTED_ROWS:
        raise ValueError(f"{name} row 수가 {EXPECTED_ROWS}가 아닙니다: {df.shape[0]}")

    if not df["timestamp"].reset_index(drop=True).equals(pd.Series(FINAL_INDEX)):
        raise ValueError(f"{name} timestamp가 FINAL_INDEX와 정확히 일치하지 않습니다.")

    if df["timestamp"].duplicated().sum() != 0:
        raise ValueError(f"{name} timestamp 중복이 있습니다.")


def read_prep_csv(filename):
    path = PREP_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"전처리 파일이 없습니다: {path}")

    df = pd.read_csv(path, encoding="utf-8-sig")

    if "timestamp" not in df.columns:
        raise KeyError(f"{filename}에 timestamp 컬럼이 없습니다.")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    if df["timestamp"].isna().any():
        raise ValueError(f"{filename}의 timestamp 파싱 실패가 있습니다.")

    return df


def save_feature(df, filename, name):
    save_path = PREP_DIR / filename
    df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print("saved:", save_path)

    saved = pd.read_csv(save_path, encoding="utf-8-sig")
    saved["timestamp"] = pd.to_datetime(saved["timestamp"])
    audit_time_index(saved, name=f"{name}_saved")
    assert_final_timestamp(saved, f"{name}_saved")
    return saved



## 1. SMP feature 생성



In [25]:
def build_smp_features():
    jeju_smp_path = resolve_raw_file("제주_시간별SMP.csv")
    mainland_smp_path = resolve_raw_file("육지_시간별SMP.csv")

    jeju_raw = clean_columns(read_csv_auto(jeju_smp_path))
    mainland_raw = clean_columns(read_csv_auto(mainland_smp_path))

    print("제주 SMP:", jeju_smp_path.name, jeju_raw.attrs.get("source_encoding"), jeju_raw.shape)
    print("육지 SMP:", mainland_smp_path.name, mainland_raw.attrs.get("source_encoding"), mainland_raw.shape)

    for name, df in [("제주", jeju_raw), ("육지", mainland_raw)]:
        period_dt = parse_date_clean(df["기간"])
        duplicated_periods = int(period_dt.duplicated().sum())
        print(f"{name} 기간 중복 row 수:", duplicated_periods)
        if duplicated_periods > 0:
            raise ValueError(f"{name} SMP 원천에 중복 기간이 있습니다.")

        boundary = df.loc[period_dt == pd.Timestamp("2024-05-31")]
        print(f"{name} 2024-05-31 row count:", len(boundary))
        if len(boundary) != 1:
            raise ValueError(f"{name} SMP에 2024-05-31 boundary row가 정확히 1개 있어야 합니다.")

        boundary_24_value = to_numeric_clean(boundary["24시"]).iloc[0]
        print(f"{name} 2024-05-31 24시 -> 2024-06-01 00:00:00 값:", boundary_24_value)

    jeju_long = wide_smp_to_long(jeju_raw, "jeju_smp")
    mainland_long = wide_smp_to_long(mainland_raw, "mainland_smp")

    audit_time_index(jeju_long, name="jeju_smp_long_raw")
    audit_time_index(mainland_long, name="mainland_smp_long_raw")

    boundary_check = (
        jeju_long
        .merge(mainland_long, on="timestamp", how="inner")
        .loc[lambda x: x["timestamp"].between("2024-06-01 00:00:00", "2024-06-01 03:00:00")]
    )
    display(boundary_check)

    boundary_row = boundary_check.loc[
        boundary_check["timestamp"] == pd.Timestamp("2024-06-01 00:00:00")
    ]
    if len(boundary_row) != 1:
        raise ValueError("2024-06-01 00:00:00 boundary row가 정확히 1개 있어야 합니다.")
    if boundary_row[["jeju_smp", "mainland_smp"]].isna().any().any():
        raise ValueError("2024-06-01 00:00:00 boundary SMP 값에 결측이 있습니다.")

    smp = align_to_final_index(jeju_long).merge(
        align_to_final_index(mainland_long),
        on="timestamp",
        how="left",
    )
    smp = smp.sort_values("timestamp").reset_index(drop=True)

    smp["is_negative_smp"] = (smp["jeju_smp"] < 0).astype("int64")
    smp["lag_smp_1h"] = smp["jeju_smp"].shift(1)
    smp["lag_smp_24h"] = smp["jeju_smp"].shift(24)
    smp["lag_smp_168h"] = smp["jeju_smp"].shift(168)
    smp["smp_rolling_24h_mean"] = smp["jeju_smp"].shift(1).rolling(window=24, min_periods=24).mean()
    smp["mainland_smp_lag_24h"] = smp["mainland_smp"].shift(24)
    smp["jeju_mainland_gap_lag_24h"] = smp["lag_smp_24h"] - smp["mainland_smp_lag_24h"]

    smp = smp[
        [
            "timestamp",
            "jeju_smp",
            "is_negative_smp",
            "lag_smp_1h",
            "lag_smp_24h",
            "lag_smp_168h",
            "smp_rolling_24h_mean",
            "mainland_smp",
            "mainland_smp_lag_24h",
            "jeju_mainland_gap_lag_24h",
        ]
    ].copy()

    audit_time_index(smp, name="smp_features_before_save")

    expected_missing = {
        "timestamp": 0,
        "jeju_smp": 0,
        "is_negative_smp": 0,
        "lag_smp_1h": 1,
        "lag_smp_24h": 24,
        "lag_smp_168h": 168,
        "smp_rolling_24h_mean": 24,
        "mainland_smp": 0,
        "mainland_smp_lag_24h": 24,
        "jeju_mainland_gap_lag_24h": 24,
    }

    missing_check = smp.isna().sum().rename("missing_count").reset_index().rename(columns={"index": "column"})
    missing_check["expected_missing"] = missing_check["column"].map(expected_missing).fillna(0).astype(int)
    missing_check["matches_expected"] = missing_check["missing_count"] == missing_check["expected_missing"]
    display(missing_check)

    unexpected_missing = missing_check.loc[~missing_check["matches_expected"]]
    if len(unexpected_missing) > 0:
        display(unexpected_missing)
        raise ValueError("SMP feature에 예상과 다른 결측이 있습니다.")

    print("마이너스 SMP 개수:", int((smp["jeju_smp"] < 0).sum()))
    print("마이너스 SMP 비율:", float((smp["jeju_smp"] < 0).mean()))

    saved = save_feature(smp, "smp_features.csv", "smp_features")
    return saved



## 2. Calendar feature 생성



In [26]:
def build_special_day_features(daily_df, window_days=3):
    daily = daily_df[["date", "holiday_type"]].copy()
    daily["date"] = pd.to_datetime(daily["date"]).dt.normalize()

    anchors = daily.loc[daily["holiday_type"] > 0, ["date", "holiday_type"]].copy()
    rows = []

    for current_date in daily["date"]:
        candidates = anchors.copy()
        candidates["special_day_offset"] = (current_date - candidates["date"]).dt.days
        candidates["abs_offset"] = candidates["special_day_offset"].abs()
        candidates = candidates.loc[candidates["abs_offset"] <= window_days].copy()

        if len(candidates) == 0:
            rows.append({
                "date": current_date,
                "special_day_window": 0,
                "special_day_type": 0,
                "special_day_offset": 0,
            })
            continue

        candidates["type_priority"] = candidates["holiday_type"].map({1: 1, 2: 1, 3: 2}).fillna(9)
        best = candidates.sort_values(["abs_offset", "type_priority", "date"]).iloc[0]

        rows.append({
            "date": current_date,
            "special_day_window": 1,
            "special_day_type": int(best["holiday_type"]),
            "special_day_offset": int(best["special_day_offset"]),
        })

    return pd.DataFrame(rows)


def make_peak_index(timestamp, is_holiday):
    hour = pd.to_datetime(timestamp).dt.hour.to_numpy()
    is_holiday_arr = pd.Series(is_holiday).astype(int).to_numpy()

    base = np.select(
        [
            (hour >= 0) & (hour <= 5),
            (hour >= 6) & (hour <= 8),
            (hour >= 9) & (hour <= 11),
            (hour >= 12) & (hour <= 16),
            (hour >= 17) & (hour <= 21),
            (hour >= 22) & (hour <= 23),
        ],
        [0.20, 0.55, 0.75, 0.40, 1.00, 0.55],
        default=0.40,
    )

    adjusted = base * np.where(is_holiday_arr == 1, 0.90, 1.00)
    return np.round(adjusted, 3)


def build_calendar_features():
    day_info_path = resolve_raw_file("day_info.csv")
    day_raw = clean_columns(read_csv_auto(day_info_path))

    print("day_info:", day_info_path.name, day_raw.attrs.get("source_encoding"), day_raw.shape)

    required_cols = [
        "날짜",
        "시간",
        "hour_sin",
        "hour_cos",
        "요일",
        "주말_유무",
        "휴일_유무",
        "휴일_종류",
    ]
    missing_cols = [c for c in required_cols if c not in day_raw.columns]
    if missing_cols:
        raise KeyError(f"day_info.csv에 필요한 컬럼이 없습니다: {missing_cols}")

    day_raw["timestamp"] = make_timestamp_from_date_time(day_raw["날짜"], day_raw["시간"])
    audit_time_index(day_raw, name="day_info_raw")

    calendar_base = day_raw.rename(columns={
        "요일": "day_of_week",
        "주말_유무": "is_weekend",
        "휴일_유무": "is_holiday",
        "휴일_종류": "holiday_type",
    }).copy()

    calendar_base = calendar_base[
        [
            "timestamp",
            "hour_sin",
            "hour_cos",
            "day_of_week",
            "is_weekend",
            "is_holiday",
            "holiday_type",
        ]
    ].copy()

    for col in ["hour_sin", "hour_cos", "day_of_week", "is_weekend", "is_holiday", "holiday_type"]:
        calendar_base[col] = pd.to_numeric(calendar_base[col], errors="coerce")

    calendar_base = align_to_final_index(calendar_base)
    audit_time_index(calendar_base, name="calendar_base")

    check = calendar_base.copy()
    expected_hour_sin = np.sin(2 * np.pi * check["timestamp"].dt.hour / 24)
    expected_hour_cos = np.cos(2 * np.pi * check["timestamp"].dt.hour / 24)
    expected_day_of_week = check["timestamp"].dt.dayofweek
    expected_is_weekend = expected_day_of_week.isin([5, 6]).astype(int)

    if not np.allclose(check["hour_sin"], expected_hour_sin, atol=1e-6):
        raise ValueError("hour_sin이 timestamp hour 기준 계산값과 다릅니다.")
    if not np.allclose(check["hour_cos"], expected_hour_cos, atol=1e-6):
        raise ValueError("hour_cos가 timestamp hour 기준 계산값과 다릅니다.")
    if not check["day_of_week"].astype(int).equals(expected_day_of_week.astype(int)):
        raise ValueError("day_of_week가 timestamp 기준 요일과 다릅니다.")
    if not check["is_weekend"].astype(int).equals(expected_is_weekend.astype(int)):
        raise ValueError("is_weekend가 timestamp 기준 주말 여부와 다릅니다.")
    if not (check.loc[check["holiday_type"] > 0, "is_holiday"] == 1).all():
        raise ValueError("holiday_type > 0인데 is_holiday가 1이 아닌 행이 있습니다.")

    daily_calendar = (
        calendar_base
        .assign(date=calendar_base["timestamp"].dt.normalize())
        .groupby("date", as_index=False)
        .agg({
            "day_of_week": "first",
            "is_weekend": "max",
            "is_holiday": "max",
            "holiday_type": "max",
        })
    )

    special_dates = daily_calendar.loc[daily_calendar["holiday_type"] > 0].copy()
    print("특수일 날짜 수:", len(special_dates))
    display(special_dates["holiday_type"].value_counts().sort_index().to_frame("date_count"))

    special_daily = build_special_day_features(daily_calendar, window_days=SPECIAL_WINDOW_DAYS)

    calendar_features = calendar_base.copy()
    calendar_features["date"] = calendar_features["timestamp"].dt.normalize()
    calendar_features = calendar_features.merge(special_daily, on="date", how="left")
    calendar_features["peak_index"] = make_peak_index(
        calendar_features["timestamp"],
        calendar_features["is_holiday"],
    )

    calendar_features = calendar_features[
        [
            "timestamp",
            "hour_sin",
            "hour_cos",
            "day_of_week",
            "is_weekend",
            "is_holiday",
            "holiday_type",
            "special_day_window",
            "special_day_type",
            "special_day_offset",
            "peak_index",
        ]
    ].copy()

    int_cols = [
        "day_of_week",
        "is_weekend",
        "is_holiday",
        "holiday_type",
        "special_day_window",
        "special_day_type",
        "special_day_offset",
    ]
    for col in int_cols:
        calendar_features[col] = calendar_features[col].astype(int)

    audit_time_index(calendar_features, name="calendar_features_before_save")

    if calendar_features.isna().sum().sum() != 0:
        display(calendar_features.isna().sum().to_frame("missing_count"))
        raise ValueError("calendar_features에 결측이 있습니다.")
    if not calendar_features["peak_index"].between(0, 1).all():
        raise ValueError("peak_index가 0~1 범위를 벗어났습니다.")

    actual_special_rows = calendar_features["holiday_type"] > 0
    if not (calendar_features.loc[actual_special_rows, "special_day_window"] == 1).all():
        raise ValueError("holiday_type > 0인 행은 special_day_window가 1이어야 합니다.")
    if not (calendar_features.loc[actual_special_rows, "special_day_offset"] == 0).all():
        raise ValueError("holiday_type > 0인 당일 행은 special_day_offset이 0이어야 합니다.")

    saved = save_feature(calendar_features, "calendar_features.csv", "calendar_features")
    if saved.isna().sum().sum() != 0:
        raise ValueError("저장본 calendar_features에 결측이 있습니다.")
    return saved



## 3. Weather feature 생성



In [27]:
def build_weather_features():
    weather_path = resolve_raw_file("기상데이터.csv")
    weather_raw = clean_columns(read_csv_auto(weather_path))

    print("기상데이터:", weather_path.name, weather_raw.attrs.get("source_encoding"), weather_raw.shape)

    required_cols = [
        "날짜",
        "시간",
        "기온(°C)",
        "강수량(mm)",
        "풍속(m/s)",
        "습도(%)",
        "일사(MJ/m2)",
        "전운량(10분위)",
    ]
    missing_cols = [c for c in required_cols if c not in weather_raw.columns]
    if missing_cols:
        raise KeyError(f"기상데이터.csv에 필요한 컬럼이 없습니다: {missing_cols}")

    weather_raw["timestamp"] = make_timestamp_from_date_time(weather_raw["날짜"], weather_raw["시간"])
    audit_time_index(weather_raw, name="weather_raw")

    weather_features = weather_raw[
        [
            "timestamp",
            "기온(°C)",
            "강수량(mm)",
            "풍속(m/s)",
            "일사(MJ/m2)",
            "전운량(10분위)",
        ]
    ].copy()

    weather_features = weather_features.rename(columns={
        "기온(°C)": "temperature_forecast",
        "강수량(mm)": "precipitation_forecast",
        "풍속(m/s)": "wind_speed_forecast",
        "일사(MJ/m2)": "solar_radiation",
        "전운량(10분위)": "cloud_cover",
    })

    feature_cols = [
        "temperature_forecast",
        "precipitation_forecast",
        "wind_speed_forecast",
        "solar_radiation",
        "cloud_cover",
    ]

    for col in feature_cols:
        weather_features[col] = to_numeric_clean(weather_features[col])

    weather_features = weather_features[
        [
            "timestamp",
            "temperature_forecast",
            "solar_radiation",
            "cloud_cover",
            "wind_speed_forecast",
            "precipitation_forecast",
        ]
    ].copy()

    weather_features = align_to_final_index(weather_features)
    audit_time_index(weather_features, name="weather_features_before_validation")

    range_checks = {
        "temperature_forecast": (-30, 45),
        "precipitation_forecast": (0, 200),
        "wind_speed_forecast": (0, 50),
        "solar_radiation": (0, 5),
        "cloud_cover": (0, 10),
    }

    range_rows = []
    for col, (lower, upper) in range_checks.items():
        range_rows.append({
            "column": col,
            "expected_min": lower,
            "expected_max": upper,
            "actual_min": weather_features[col].min(),
            "actual_max": weather_features[col].max(),
            "below_count": int((weather_features[col] < lower).sum()),
            "above_count": int((weather_features[col] > upper).sum()),
        })

    range_check = pd.DataFrame(range_rows)
    display(range_check)
    bad_range = range_check.loc[(range_check["below_count"] > 0) | (range_check["above_count"] > 0)]
    if len(bad_range) > 0:
        raise ValueError("기상 feature 값 범위에 이상치가 있습니다.")

    if weather_features.isna().sum().sum() != 0:
        display(weather_features.isna().sum().to_frame("missing_count"))
        raise ValueError("weather_features에 결측이 있습니다.")

    night_solar_max = weather_features.loc[
        weather_features["timestamp"].dt.hour.isin([0, 1, 2, 3, 4, 5, 22, 23]),
        "solar_radiation",
    ].max()
    print("야간 시간대 solar_radiation max:", night_solar_max)

    saved = save_feature(weather_features, "weather_features.csv", "weather_features")
    if saved.isna().sum().sum() != 0:
        raise ValueError("저장본 weather_features에 결측이 있습니다.")
    return saved



## 4. Renewable actual 생성



In [28]:
def build_renewable_actual():
    renewable_specs = [
        {
            "year": 2024,
            "file": "지역별 시간별 태양광 풍력 발전량_2024.csv",
            "date_col": "거래일자",
        },
        {
            "year": 2025,
            "file": "지역별 시간별 태양광 및 풍력 발전량_2025.csv",
            "date_col": "거래일",
        },
    ]

    raw_frames = []
    for spec in renewable_specs:
        path = resolve_raw_file(spec["file"])
        df = clean_columns(read_csv_auto(path))
        df["source_year"] = spec["year"]
        print(f'{spec["year"]} renewable:', path.name, df.attrs.get("source_encoding"), df.shape)
        raw_frames.append(df)

    renewable_raw = pd.concat(raw_frames, ignore_index=True)
    print("통합 renewable_raw shape:", renewable_raw.shape)

    required_cols_common = ["거래시간", "지역", "연료원", "전력거래량(MWh)", "source_year"]
    for col in required_cols_common:
        if col not in renewable_raw.columns:
            raise KeyError(f"재생에너지 원천에 필요한 컬럼이 없습니다: {col}")

    jeju_region_values = sorted([
        x for x in renewable_raw["지역"].dropna().astype(str).unique()
        if "제주" in x
    ])
    print("지역 목록 중 제주 포함 값:", jeju_region_values)
    display(renewable_raw["연료원"].value_counts().to_frame("row_count"))

    renewable_jeju = renewable_raw[
        renewable_raw["지역"].astype(str).str.contains("제주", na=False)
        & renewable_raw["연료원"].isin(["태양광", "풍력"])
    ].copy()
    print("renewable_jeju shape:", renewable_jeju.shape)

    renewable_jeju["date_raw"] = pd.Series(pd.NA, index=renewable_jeju.index, dtype="object")
    mask_2024 = renewable_jeju["source_year"] == 2024
    mask_2025 = renewable_jeju["source_year"] == 2025
    renewable_jeju.loc[mask_2024, "date_raw"] = renewable_jeju.loc[mask_2024, "거래일자"]
    renewable_jeju.loc[mask_2025, "date_raw"] = renewable_jeju.loc[mask_2025, "거래일"]

    if renewable_jeju["date_raw"].isna().any():
        display(renewable_jeju.loc[renewable_jeju["date_raw"].isna()].head())
        raise ValueError("date_raw 생성 실패 행이 있습니다.")

    renewable_jeju["timestamp"] = make_timestamp_hour_ending(
        renewable_jeju["date_raw"],
        renewable_jeju["거래시간"],
    )
    renewable_jeju["generation_mwh"] = to_numeric_clean(renewable_jeju["전력거래량(MWh)"])

    if renewable_jeju["generation_mwh"].isna().any():
        display(renewable_jeju.loc[renewable_jeju["generation_mwh"].isna()].head())
        raise ValueError("전력거래량(MWh) 숫자 변환 실패가 있습니다.")

    audit_time_index(renewable_jeju[["timestamp", "generation_mwh"]], name="renewable_jeju_long_raw")

    renewable_pivot = (
        renewable_jeju
        .pivot_table(
            index="timestamp",
            columns="연료원",
            values="generation_mwh",
            aggfunc="sum",
        )
        .reset_index()
    )
    renewable_pivot.columns.name = None
    renewable_pivot = renewable_pivot.rename(columns={
        "태양광": "actual_solar",
        "풍력": "actual_wind",
    })

    for col in ["actual_solar", "actual_wind"]:
        if col not in renewable_pivot.columns:
            renewable_pivot[col] = np.nan

    renewable_pivot = renewable_pivot[["timestamp", "actual_solar", "actual_wind"]].copy()
    renewable_pivot["actual_renewable_total"] = renewable_pivot["actual_solar"] + renewable_pivot["actual_wind"]
    renewable_pivot = renewable_pivot.sort_values("timestamp").reset_index(drop=True)

    audit_time_index(renewable_pivot, name="renewable_pivot_raw")

    renewable_actual = align_to_final_index(renewable_pivot)
    renewable_actual["has_renewable_actual"] = (
        renewable_actual[["actual_solar", "actual_wind"]].notna().all(axis=1).astype(int)
    )
    renewable_actual = renewable_actual[
        [
            "timestamp",
            "actual_solar",
            "actual_wind",
            "actual_renewable_total",
            "has_renewable_actual",
        ]
    ].copy()

    audit_time_index(renewable_actual, name="renewable_actual_before_save")

    print("has_renewable_actual 분포")
    display(renewable_actual["has_renewable_actual"].value_counts().sort_index().to_frame("row_count"))
    print("actual 데이터가 있는 마지막 timestamp:", renewable_actual.loc[renewable_actual["has_renewable_actual"] == 1, "timestamp"].max())
    print("actual 데이터가 없는 첫 timestamp:", renewable_actual.loc[renewable_actual["has_renewable_actual"] == 0, "timestamp"].min())

    actual_missing_rows = renewable_actual.loc[renewable_actual["has_renewable_actual"] == 0].copy()
    expected_missing_start = pd.Timestamp("2026-01-01 01:00:00")
    if len(actual_missing_rows) > 0:
        actual_missing_start = actual_missing_rows["timestamp"].min()
        actual_missing_end = actual_missing_rows["timestamp"].max()
        print("actual missing start:", actual_missing_start)
        print("actual missing end  :", actual_missing_end)
        if actual_missing_start != expected_missing_start:
            raise ValueError(
                f"예상한 actual 결측 시작은 {expected_missing_start}인데 실제는 {actual_missing_start}입니다."
            )

    available = renewable_actual.loc[renewable_actual["has_renewable_actual"] == 1].copy()
    negative_counts = {
        "actual_solar_negative": int((available["actual_solar"] < 0).sum()),
        "actual_wind_negative": int((available["actual_wind"] < 0).sum()),
        "actual_renewable_total_negative": int((available["actual_renewable_total"] < 0).sum()),
    }
    display(pd.DataFrame([negative_counts]))
    if any(v > 0 for v in negative_counts.values()):
        raise ValueError("재생에너지 실제 발전량에 음수 값이 있습니다.")

    available_missing = renewable_actual.loc[
        renewable_actual["has_renewable_actual"] == 1,
        ["actual_solar", "actual_wind", "actual_renewable_total"],
    ].isna().sum()
    display(available_missing.to_frame("missing_count_when_available"))
    if available_missing.sum() != 0:
        raise ValueError("has_renewable_actual=1인 구간에 actual 값 결측이 있습니다.")

    saved = save_feature(renewable_actual, "renewable_actual.csv", "renewable_actual")
    return saved



## 5. Renewable forecast 생성

`actual_solar`, `actual_wind`를 Model B feature로 직접 쓰지 않는다.
기상/달력 기반 보조모델을 학습해 `forecast_solar`, `forecast_wind`를 만든다.



In [29]:
def make_time_features(df):
    out = df.copy()
    ts = out["timestamp"]

    out["hour"] = ts.dt.hour
    out["month"] = ts.dt.month
    out["dayofyear"] = ts.dt.dayofyear
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    out["dayofyear_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 366)
    out["dayofyear_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 366)
    out["wind_speed_squared"] = out["wind_speed_forecast"] ** 2
    out["wind_speed_cubed"] = out["wind_speed_forecast"] ** 3
    out["is_daylight_proxy"] = (
        (out["solar_radiation"] > 0.01)
        | out["hour"].between(6, 19)
    ).astype(int)

    return out


def make_model():
    from sklearn.ensemble import HistGradientBoostingRegressor

    return HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=350,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )


def regression_metrics(y_true, y_pred, name):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    mask = pd.Series(y_true).notna() & pd.Series(y_pred).notna()
    y = pd.Series(y_true).loc[mask].astype(float)
    pred = pd.Series(y_pred).loc[mask].astype(float)

    return {
        "target": name,
        "n": int(mask.sum()),
        "mae": mean_absolute_error(y, pred),
        "rmse": np.sqrt(mean_squared_error(y, pred)),
        "r2": r2_score(y, pred),
    }


def make_walk_forward_oof(df, target_col, feature_cols, min_train_days=60, valid_days=30):
    work = df.sort_values("timestamp").copy()
    actual_mask = work[target_col].notna()

    actual_start = work.loc[actual_mask, "timestamp"].min()
    actual_end = work.loc[actual_mask, "timestamp"].max()

    val_start = actual_start + pd.Timedelta(days=min_train_days)
    valid_delta = pd.Timedelta(days=valid_days)

    oof_pred = pd.Series(np.nan, index=work.index, dtype=float)
    fold_rows = []
    fold_no = 1

    while val_start <= actual_end:
        val_end_exclusive = val_start + valid_delta

        train_mask = actual_mask & (work["timestamp"] < val_start)
        val_mask = (
            actual_mask
            & (work["timestamp"] >= val_start)
            & (work["timestamp"] < val_end_exclusive)
        )

        train_idx = work.index[train_mask]
        val_idx = work.index[val_mask]

        if len(val_idx) == 0:
            val_start = val_end_exclusive
            continue

        if len(train_idx) < 24 * 14:
            raise ValueError("학습 데이터가 너무 적습니다. MIN_TRAIN_DAYS를 늘리거나 데이터를 확인하세요.")

        model = make_model()
        model.fit(work.loc[train_idx, feature_cols], work.loc[train_idx, target_col])

        pred = model.predict(work.loc[val_idx, feature_cols])
        pred = np.clip(pred, 0, None)
        oof_pred.loc[val_idx] = pred

        fold_rows.append({
            "target": target_col,
            "fold": fold_no,
            "train_start": work.loc[train_idx, "timestamp"].min(),
            "train_end": work.loc[train_idx, "timestamp"].max(),
            "valid_start": work.loc[val_idx, "timestamp"].min(),
            "valid_end": work.loc[val_idx, "timestamp"].max(),
            "train_rows": len(train_idx),
            "valid_rows": len(val_idx),
        })

        fold_no += 1
        val_start = val_end_exclusive

    fold_info = pd.DataFrame(fold_rows)

    final_model = make_model()
    final_train_idx = work.index[actual_mask]
    final_model.fit(work.loc[final_train_idx, feature_cols], work.loc[final_train_idx, target_col])

    final_pred = pd.Series(
        np.clip(final_model.predict(work[feature_cols]), 0, None),
        index=work.index,
        dtype=float,
    )

    return oof_pred, final_pred, fold_info


def build_renewable_forecast():
    weather = read_prep_csv("weather_features.csv")
    calendar = read_prep_csv("calendar_features.csv")
    renewable_actual = read_prep_csv("renewable_actual.csv")

    for name, df in [
        ("weather_features", weather),
        ("calendar_features", calendar),
        ("renewable_actual", renewable_actual),
    ]:
        print(name)
        audit_time_index(df, name=name)
        assert_final_timestamp(df.sort_values("timestamp").reset_index(drop=True), name)

    modeling = (
        pd.DataFrame({"timestamp": FINAL_INDEX})
        .merge(weather, on="timestamp", how="left")
        .merge(calendar, on="timestamp", how="left")
        .merge(renewable_actual, on="timestamp", how="left")
    )
    modeling = make_time_features(modeling)
    audit_time_index(modeling, name="renewable_forecast_modeling_table")

    required_input_cols = [
        "temperature_forecast",
        "solar_radiation",
        "cloud_cover",
        "wind_speed_forecast",
        "precipitation_forecast",
        "hour_sin",
        "hour_cos",
        "day_of_week",
        "is_weekend",
        "is_holiday",
        "holiday_type",
        "special_day_window",
        "special_day_type",
        "special_day_offset",
        "peak_index",
        "hour",
        "month",
        "dayofyear",
        "month_sin",
        "month_cos",
        "dayofyear_sin",
        "dayofyear_cos",
        "wind_speed_squared",
        "wind_speed_cubed",
        "is_daylight_proxy",
    ]

    missing_input_cols = [c for c in required_input_cols if c not in modeling.columns]
    if missing_input_cols:
        raise KeyError(f"모델 입력 컬럼이 없습니다: {missing_input_cols}")

    input_missing = modeling[required_input_cols].isna().sum()
    display(input_missing[input_missing > 0].to_frame("missing_count"))
    if input_missing.sum() != 0:
        raise ValueError("재생에너지 예측 입력 feature에 결측이 있습니다.")

    feature_cols = required_input_cols

    solar_oof, solar_final_pred, solar_folds = make_walk_forward_oof(
        modeling,
        target_col="actual_solar",
        feature_cols=feature_cols,
        min_train_days=MIN_TRAIN_DAYS,
        valid_days=VALID_DAYS,
    )
    wind_oof, wind_final_pred, wind_folds = make_walk_forward_oof(
        modeling,
        target_col="actual_wind",
        feature_cols=feature_cols,
        min_train_days=MIN_TRAIN_DAYS,
        valid_days=VALID_DAYS,
    )

    print("solar folds")
    display(solar_folds)
    print("wind folds")
    display(wind_folds)

    modeling["forecast_solar_oof"] = solar_oof
    modeling["forecast_wind_oof"] = wind_oof
    modeling["forecast_solar_final_model"] = solar_final_pred
    modeling["forecast_wind_final_model"] = wind_final_pred

    zero_solar_mask = (
        modeling["timestamp"].dt.hour.isin([0, 1, 2, 3, 4, 5, 22, 23])
        & (modeling["solar_radiation"] <= 0.001)
    )

    for col in ["forecast_solar_oof", "forecast_solar_final_model"]:
        modeling.loc[zero_solar_mask, col] = 0

    metrics_df = pd.DataFrame([
        regression_metrics(modeling["actual_solar"], modeling["forecast_solar_oof"], "actual_solar_oof"),
        regression_metrics(modeling["actual_wind"], modeling["forecast_wind_oof"], "actual_wind_oof"),
    ])
    display(metrics_df)

    forecast = modeling[["timestamp", "has_renewable_actual"]].copy()
    forecast["forecast_solar"] = modeling["forecast_solar_oof"]
    forecast["forecast_wind"] = modeling["forecast_wind_oof"]

    solar_oof_available = forecast["forecast_solar"].notna()
    wind_oof_available = forecast["forecast_wind"].notna()
    both_oof_available = solar_oof_available & wind_oof_available

    forecast.loc[~solar_oof_available, "forecast_solar"] = modeling.loc[
        ~solar_oof_available,
        "forecast_solar_final_model",
    ]
    forecast.loc[~wind_oof_available, "forecast_wind"] = modeling.loc[
        ~wind_oof_available,
        "forecast_wind_final_model",
    ]

    forecast["renewable_forecast_source"] = np.where(
        both_oof_available,
        "walk_forward_oof",
        np.where(
            forecast["has_renewable_actual"] == 1,
            "final_model_initial_backfill",
            "final_model_no_actual",
        ),
    )
    forecast["renewable_forecast_oof_flag"] = both_oof_available.astype(int)

    forecast["forecast_solar"] = forecast["forecast_solar"].clip(lower=0)
    forecast["forecast_wind"] = forecast["forecast_wind"].clip(lower=0)
    forecast["renewable_pen"] = np.nan
    forecast["net_load"] = np.nan

    renewable_forecast = forecast[
        [
            "timestamp",
            "forecast_solar",
            "forecast_wind",
            "renewable_pen",
            "net_load",
        ]
    ].copy()

    audit_time_index(renewable_forecast, name="renewable_forecast_before_save")

    print("forecast source 분포")
    display(forecast["renewable_forecast_source"].value_counts().to_frame("row_count"))
    print("OOF flag 분포")
    display(forecast["renewable_forecast_oof_flag"].value_counts().sort_index().to_frame("row_count"))

    non_null_missing = renewable_forecast[["forecast_solar", "forecast_wind"]].isna().sum()
    display(non_null_missing.to_frame("missing_count"))
    if non_null_missing.sum() != 0:
        raise ValueError("forecast_solar 또는 forecast_wind에 결측이 있습니다.")
    if (renewable_forecast["forecast_solar"] < 0).any():
        raise ValueError("forecast_solar에 음수 예측값이 있습니다.")
    if (renewable_forecast["forecast_wind"] < 0).any():
        raise ValueError("forecast_wind에 음수 예측값이 있습니다.")

    placeholder_missing = renewable_forecast[["renewable_pen", "net_load"]].isna().sum()
    display(placeholder_missing.to_frame("missing_count"))
    if placeholder_missing["renewable_pen"] != EXPECTED_ROWS:
        raise ValueError("predicted_demand가 없으므로 renewable_pen은 현재 전부 NaN이어야 합니다.")
    if placeholder_missing["net_load"] != EXPECTED_ROWS:
        raise ValueError("predicted_demand가 없으므로 net_load는 현재 전부 NaN이어야 합니다.")

    night_solar_max = renewable_forecast.loc[
        renewable_forecast["timestamp"].dt.hour.isin([0, 1, 2, 3, 4, 5, 22, 23]),
        "forecast_solar",
    ].max()
    print("야간 forecast_solar max:", night_solar_max)

    saved = save_feature(renewable_forecast, "renewable_forecast.csv", "renewable_forecast")
    if saved[["forecast_solar", "forecast_wind"]].isna().sum().sum() != 0:
        raise ValueError("저장본 forecast_solar 또는 forecast_wind에 결측이 있습니다.")
    return saved



## 6. Fuel cost feature 생성



In [30]:
def build_fuel_cost_features():
    fuel_path = resolve_raw_file("연료비용.csv")
    fuel_raw = clean_columns(read_csv_auto(fuel_path))

    print("연료비용:", fuel_path.name, fuel_raw.attrs.get("source_encoding"), fuel_raw.shape)

    if fuel_raw.shape[0] < 3:
        raise ValueError("연료비용.csv는 최소 3행 이상이어야 합니다.")

    month_col = fuel_raw.columns[0]
    lng_fuel_price_col = find_fuel_column(fuel_raw, "연료단가", "LNG")
    lng_heat_price_col = find_fuel_column(fuel_raw, "열량단가", "LNG")
    lng_fuel_cost_unit_col = find_fuel_column(fuel_raw, "연료비단가", "LNG")

    print("월 컬럼:", month_col)
    print("LNG 연료단가 컬럼:", lng_fuel_price_col)
    print("LNG 열량단가 컬럼:", lng_heat_price_col)
    print("LNG 연료비단가 컬럼:", lng_fuel_cost_unit_col)

    fuel_monthly = fuel_raw.iloc[2:].copy()
    fuel_monthly["month_start"] = fuel_monthly[month_col].apply(parse_epsis_month)
    if fuel_monthly["month_start"].isna().any():
        display(fuel_monthly.loc[fuel_monthly["month_start"].isna()].head(10))
        raise ValueError("월 표기 파싱 실패가 있습니다.")

    fuel_monthly["lng_fuel_price_krw_per_ton"] = to_numeric_clean(fuel_monthly[lng_fuel_price_col])
    fuel_monthly["lng_heat_price_krw_per_gcal"] = to_numeric_clean(fuel_monthly[lng_heat_price_col])
    fuel_monthly["fuel_cost_unit"] = to_numeric_clean(fuel_monthly[lng_fuel_cost_unit_col])

    required_numeric = [
        "lng_fuel_price_krw_per_ton",
        "lng_heat_price_krw_per_gcal",
        "fuel_cost_unit",
    ]
    if fuel_monthly[required_numeric].isna().any().any():
        display(fuel_monthly.loc[fuel_monthly[required_numeric].isna().any(axis=1)])
        raise ValueError("LNG 연료비용 숫자 변환 실패가 있습니다.")

    fuel_monthly = fuel_monthly.sort_values("month_start").reset_index(drop=True)
    if fuel_monthly["month_start"].duplicated().sum() != 0:
        duplicated = fuel_monthly.loc[fuel_monthly["month_start"].duplicated(), "month_start"]
        raise ValueError(f"중복 월이 있습니다: {duplicated.tolist()}")

    baseline_rows = fuel_monthly.loc[fuel_monthly["month_start"] == BASELINE_MONTH]
    if len(baseline_rows) != 1:
        raise ValueError(f"lng_price_index 기준월 {BASELINE_MONTH.date()} 행이 정확히 1개 있어야 합니다.")

    baseline_lng_price = baseline_rows["lng_fuel_price_krw_per_ton"].iloc[0]
    fuel_monthly["lng_price_index"] = fuel_monthly["lng_fuel_price_krw_per_ton"] / baseline_lng_price * 100

    fuel_monthly_features = fuel_monthly[
        [
            "month_start",
            "lng_fuel_price_krw_per_ton",
            "lng_heat_price_krw_per_gcal",
            "lng_price_index",
            "fuel_cost_unit",
        ]
    ].copy()
    display(fuel_monthly_features)
    print("baseline_lng_price:", baseline_lng_price)

    required_months = pd.date_range(
        FINAL_START.normalize().replace(day=1),
        FINAL_END.normalize().replace(day=1),
        freq="MS",
    )
    available_months = pd.DatetimeIndex(fuel_monthly_features["month_start"])
    missing_months = required_months.difference(available_months)
    print("required_months:", len(required_months), required_months.min(), required_months.max())
    print("available fuel months:", len(available_months), available_months.min(), available_months.max())
    print("missing_months:", list(missing_months))
    if len(missing_months) > 0:
        raise ValueError(f"최종 기간에 필요한 월별 연료비용 데이터가 없습니다: {list(missing_months)}")

    hourly_base = pd.DataFrame({"timestamp": FINAL_INDEX})
    monthly_for_asof = (
        fuel_monthly_features[
            [
                "month_start",
                "lng_price_index",
                "fuel_cost_unit",
            ]
        ]
        .rename(columns={"month_start": "fuel_cost_effective_month"})
        .sort_values("fuel_cost_effective_month")
        .reset_index(drop=True)
    )

    fuel_hourly = pd.merge_asof(
        hourly_base.sort_values("timestamp"),
        monthly_for_asof,
        left_on="timestamp",
        right_on="fuel_cost_effective_month",
        direction="backward",
    )

    audit_time_index(fuel_hourly, name="fuel_hourly_with_effective_month")

    fuel_cost_features = fuel_hourly[
        [
            "timestamp",
            "lng_price_index",
            "fuel_cost_unit",
        ]
    ].copy()
    audit_time_index(fuel_cost_features, name="fuel_cost_features_before_save")

    if fuel_cost_features.isna().sum().sum() != 0:
        display(fuel_cost_features.isna().sum().to_frame("missing_count"))
        raise ValueError("fuel_cost_features에 결측이 있습니다.")
    if (fuel_cost_features["lng_price_index"] <= 0).any():
        raise ValueError("lng_price_index에 0 이하 값이 있습니다.")
    if (fuel_cost_features["fuel_cost_unit"] <= 0).any():
        raise ValueError("fuel_cost_unit에 0 이하 값이 있습니다.")

    saved = save_feature(fuel_cost_features, "fuel_cost_features.csv", "fuel_cost_features")
    if saved.isna().sum().sum() != 0:
        raise ValueError("저장본 fuel_cost_features에 결측이 있습니다.")
    return saved



## 7. Oil / USD feature 생성



In [31]:
def build_oil_usd_features():
    oil_path = resolve_raw_file("국제_원유가격(Dubai).csv")
    oil_raw = clean_columns(read_csv_auto(oil_path))

    print("Dubai oil:", oil_path.name, oil_raw.attrs.get("source_encoding"), oil_raw.shape)

    required_oil_cols = ["기간", "Dubai"]
    missing_cols = [c for c in required_oil_cols if c not in oil_raw.columns]
    if missing_cols:
        raise KeyError(f"국제_원유가격(Dubai).csv에 필요한 컬럼이 없습니다: {missing_cols}")

    oil_daily = oil_raw[["기간", "Dubai"]].copy()
    oil_daily["date"] = parse_korean_2digit_date(oil_daily["기간"])
    oil_daily["oil_price_dubai"] = to_numeric_clean(oil_daily["Dubai"])

    raw_oil_missing = oil_daily.loc[oil_daily["oil_price_dubai"].isna()].copy()
    print("Dubai 원천 결측 row 수:", len(raw_oil_missing))
    if len(raw_oil_missing) > 0:
        display(raw_oil_missing.head(20))

    oil_daily = (
        oil_daily[["date", "oil_price_dubai"]]
        .drop_duplicates(subset=["date"], keep="last")
        .sort_values("date")
        .reset_index(drop=True)
    )
    if oil_daily["oil_price_dubai"].notna().sum() == 0:
        raise ValueError("Dubai 유가의 유효 숫자값이 하나도 없습니다.")

    first_valid_oil_date = oil_daily.loc[oil_daily["oil_price_dubai"].notna(), "date"].min()

    print("oil_daily date range:", oil_daily["date"].min(), "~", oil_daily["date"].max())
    print("oil_daily rows:", len(oil_daily))
    print("first_valid_oil_date:", first_valid_oil_date)

    daily_index = pd.date_range(FINAL_START.normalize(), FINAL_END.normalize(), freq="D")
    oil_daily_grid = pd.DataFrame({"date": daily_index})
    oil_daily_grid = oil_daily_grid.merge(oil_daily, on="date", how="left")
    oil_daily_grid["oil_price_dubai"] = oil_daily_grid["oil_price_dubai"].ffill()

    leading_oil_missing_days = oil_daily_grid.loc[oil_daily_grid["oil_price_dubai"].isna(), "date"]
    print("leading_oil_missing_days:", leading_oil_missing_days.tolist())

    hourly_base = pd.DataFrame({"timestamp": FINAL_INDEX})
    hourly_base["date"] = hourly_base["timestamp"].dt.normalize()
    oil_hourly = hourly_base.merge(oil_daily_grid, on="date", how="left")
    oil_hourly = oil_hourly[["timestamp", "oil_price_dubai"]].copy()
    audit_time_index(oil_hourly, name="oil_hourly")

    usd_path = resolve_raw_file("USD_KRW.csv")
    usd_raw = clean_columns(read_csv_auto(usd_path))
    print("USD_KRW:", usd_path.name, usd_raw.attrs.get("source_encoding"), usd_raw.shape)

    required_usd_cols = ["date", "usd_krw_filled"]
    missing_cols = [c for c in required_usd_cols if c not in usd_raw.columns]
    if missing_cols:
        raise KeyError(f"USD_KRW.csv에 필요한 컬럼이 없습니다: {missing_cols}")

    usd_daily = usd_raw[["date", "usd_krw_filled"]].copy()
    usd_daily["date"] = pd.to_datetime(usd_daily["date"], errors="coerce").dt.normalize()
    usd_daily["usd_krw"] = to_numeric_clean(usd_daily["usd_krw_filled"])

    if usd_daily["date"].isna().any():
        display(usd_daily.loc[usd_daily["date"].isna()].head())
        raise ValueError("USD_KRW date 파싱 실패가 있습니다.")
    if usd_daily["usd_krw"].isna().any():
        display(usd_daily.loc[usd_daily["usd_krw"].isna()].head())
        raise ValueError("usd_krw_filled 숫자 변환 실패가 있습니다.")

    usd_daily = (
        usd_daily[["date", "usd_krw"]]
        .drop_duplicates(subset=["date"], keep="last")
        .sort_values("date")
        .reset_index(drop=True)
    )

    usd_daily_grid = pd.DataFrame({"date": daily_index})
    usd_daily_grid = usd_daily_grid.merge(usd_daily, on="date", how="left")
    missing_usd_days = usd_daily_grid.loc[usd_daily_grid["usd_krw"].isna(), "date"]
    print("missing_usd_days:", missing_usd_days.tolist())
    if len(missing_usd_days) > 0:
        raise ValueError("최종 기간 내 USD_KRW 일별 데이터 결측이 있습니다.")

    usd_hourly = hourly_base[["timestamp", "date"]].merge(usd_daily_grid, on="date", how="left")
    usd_hourly = usd_hourly[["timestamp", "usd_krw"]].copy()
    audit_time_index(usd_hourly, name="usd_hourly")

    oil_usd_features = (
        pd.DataFrame({"timestamp": FINAL_INDEX})
        .merge(oil_hourly, on="timestamp", how="left")
        .merge(usd_hourly, on="timestamp", how="left")
    )
    audit_time_index(oil_usd_features, name="oil_usd_features_before_save")

    expected_oil_missing_hours = int(max(0, (first_valid_oil_date - FINAL_START.normalize()).days * 24))
    actual_oil_missing_hours = int(oil_usd_features["oil_price_dubai"].isna().sum())
    print("expected_oil_missing_hours:", expected_oil_missing_hours)
    print("actual_oil_missing_hours  :", actual_oil_missing_hours)
    if actual_oil_missing_hours != expected_oil_missing_hours:
        raise ValueError(
            f"oil_price_dubai 결측 수가 예상과 다릅니다. expected={expected_oil_missing_hours}, actual={actual_oil_missing_hours}"
        )
    if oil_usd_features["usd_krw"].isna().sum() != 0:
        raise ValueError("usd_krw에 결측이 있습니다.")

    available_oil = oil_usd_features["oil_price_dubai"].dropna()
    if (available_oil <= 0).any():
        raise ValueError("oil_price_dubai에 0 이하 값이 있습니다.")
    if (oil_usd_features["usd_krw"] <= 0).any():
        raise ValueError("usd_krw에 0 이하 값이 있습니다.")

    saved = save_feature(oil_usd_features, "oil_usd_features.csv", "oil_usd_features")
    if saved["usd_krw"].isna().sum() != 0:
        raise ValueError("저장본 usd_krw 결측이 있습니다.")
    if saved["oil_price_dubai"].isna().sum() != expected_oil_missing_hours:
        raise ValueError("저장본 oil_price_dubai 결측 수가 예상과 다릅니다.")
    return saved



## 8. 최종 `model_b_data.csv` 병합



In [32]:
def merge_model_b_data():
    prep_files = {
        "smp": "smp_features.csv",
        "calendar": "calendar_features.csv",
        "weather": "weather_features.csv",
        "renewable": "renewable_forecast.csv",
        "fuel": "fuel_cost_features.csv",
        "oil_usd": "oil_usd_features.csv",
    }

    loaded = {}
    for key, filename in prep_files.items():
        df = read_prep_csv(filename).sort_values("timestamp").reset_index(drop=True)
        print(f"\n[{key}] {filename}")
        audit_time_index(df, name=filename)
        assert_final_timestamp(df, filename)
        loaded[key] = df

    model_b = pd.DataFrame({"timestamp": FINAL_INDEX})
    for key in ["smp", "calendar", "weather", "renewable", "fuel", "oil_usd"]:
        before_cols = set(model_b.columns)
        model_b = model_b.merge(
            loaded[key],
            on="timestamp",
            how="left",
            validate="one_to_one",
        )
        added_cols = [c for c in model_b.columns if c not in before_cols]
        print(f"{key} merged. added columns:", added_cols)

    audit_time_index(model_b, name="model_b_merged_raw")
    assert_final_timestamp(model_b, "model_b_merged_raw")

    if "predicted_demand" not in model_b.columns:
        model_b["predicted_demand"] = np.nan
    if "renewable_pen" not in model_b.columns:
        model_b["renewable_pen"] = np.nan
    if "net_load" not in model_b.columns:
        model_b["net_load"] = np.nan

    model_b_final = model_b.copy()
    model_b_final.insert(1, "날짜", model_b_final["timestamp"].dt.strftime("%Y-%m-%d"))
    model_b_final.insert(2, "시간", model_b_final["timestamp"].dt.strftime("%H:%M:%S"))

    final_columns = [
        "timestamp",
        "날짜",
        "시간",
        "jeju_smp",
        "is_negative_smp",
        "hour_sin",
        "hour_cos",
        "day_of_week",
        "is_weekend",
        "is_holiday",
        "holiday_type",
        "special_day_window",
        "special_day_type",
        "special_day_offset",
        "peak_index",
        "lag_smp_1h",
        "lag_smp_24h",
        "lag_smp_168h",
        "smp_rolling_24h_mean",
        "mainland_smp",
        "mainland_smp_lag_24h",
        "jeju_mainland_gap_lag_24h",
        "predicted_demand",
        "temperature_forecast",
        "solar_radiation",
        "cloud_cover",
        "wind_speed_forecast",
        "precipitation_forecast",
        "forecast_solar",
        "forecast_wind",
        "renewable_pen",
        "net_load",
        "lng_price_index",
        "fuel_cost_unit",
        "oil_price_dubai",
        "usd_krw",
    ]

    missing_final_cols = [c for c in final_columns if c not in model_b_final.columns]
    extra_cols = [c for c in model_b_final.columns if c not in final_columns]
    print("missing_final_cols:", missing_final_cols)
    print("extra_cols:", extra_cols)
    if missing_final_cols:
        raise KeyError(f"최종 컬럼 목록에 필요한 컬럼이 없습니다: {missing_final_cols}")
    if extra_cols:
        print("주의: 최종 목록에 없는 컬럼은 제외합니다:", extra_cols)

    model_b_final = model_b_final[final_columns].copy()
    audit_time_index(model_b_final, name="model_b_final_before_validation")

    expected_missing = {
        "timestamp": 0,
        "날짜": 0,
        "시간": 0,
        "jeju_smp": 0,
        "is_negative_smp": 0,
        "hour_sin": 0,
        "hour_cos": 0,
        "day_of_week": 0,
        "is_weekend": 0,
        "is_holiday": 0,
        "holiday_type": 0,
        "special_day_window": 0,
        "special_day_type": 0,
        "special_day_offset": 0,
        "peak_index": 0,
        "lag_smp_1h": 1,
        "lag_smp_24h": 24,
        "lag_smp_168h": 168,
        "smp_rolling_24h_mean": 24,
        "mainland_smp": 0,
        "mainland_smp_lag_24h": 24,
        "jeju_mainland_gap_lag_24h": 24,
        "predicted_demand": EXPECTED_ROWS,
        "temperature_forecast": 0,
        "solar_radiation": 0,
        "cloud_cover": 0,
        "wind_speed_forecast": 0,
        "precipitation_forecast": 0,
        "forecast_solar": 0,
        "forecast_wind": 0,
        "renewable_pen": EXPECTED_ROWS,
        "net_load": EXPECTED_ROWS,
        "lng_price_index": 0,
        "fuel_cost_unit": 0,
        "usd_krw": 0,
    }

    oil_missing_count = int(model_b_final["oil_price_dubai"].isna().sum())
    expected_missing["oil_price_dubai"] = oil_missing_count

    missing_check = (
        model_b_final.isna().sum()
        .rename("missing_count")
        .reset_index()
        .rename(columns={"index": "column"})
    )
    missing_check["expected_missing"] = missing_check["column"].map(expected_missing).fillna(-1).astype(int)
    missing_check["matches_expected"] = missing_check["missing_count"] == missing_check["expected_missing"]
    display(missing_check)

    unexpected_missing = missing_check.loc[~missing_check["matches_expected"]]
    if len(unexpected_missing) > 0:
        display(unexpected_missing)
        raise ValueError("예상과 다른 결측 패턴이 있습니다.")

    oil_missing_mask = model_b_final["oil_price_dubai"].isna()
    if oil_missing_count > 0:
        oil_missing_timestamps = model_b_final.loc[oil_missing_mask, "timestamp"]
        expected_leading_missing = pd.date_range(FINAL_START, periods=oil_missing_count, freq="h")
        if not oil_missing_timestamps.reset_index(drop=True).equals(pd.Series(expected_leading_missing)):
            raise ValueError("oil_price_dubai 결측이 초기 연속 구간이 아닙니다.")

    if not ((model_b_final["jeju_smp"] < 0).astype(int).equals(model_b_final["is_negative_smp"].astype(int))):
        raise ValueError("is_negative_smp가 jeju_smp < 0 조건과 일치하지 않습니다.")

    target_cols = ["jeju_smp", "is_negative_smp"]
    leakage_risk_cols = ["mainland_smp"]
    placeholder_cols = ["predicted_demand", "renewable_pen", "net_load"]
    recommended_exclude_from_training = (
        target_cols
        + leakage_risk_cols
        + placeholder_cols
        + ["timestamp", "날짜", "시간"]
    )
    recommended_feature_cols = [
        c for c in model_b_final.columns
        if c not in recommended_exclude_from_training
    ]

    print("target_cols")
    display(pd.DataFrame({"column": target_cols}))
    print("leakage_risk_cols")
    display(pd.DataFrame({"column": leakage_risk_cols}))
    print("placeholder_cols")
    display(pd.DataFrame({"column": placeholder_cols}))
    print("recommended_feature_cols")
    display(pd.DataFrame({"column": recommended_feature_cols}))
    print("recommended_feature_count:", len(recommended_feature_cols))

    range_cols = [
        "jeju_smp",
        "mainland_smp",
        "forecast_solar",
        "forecast_wind",
        "temperature_forecast",
        "solar_radiation",
        "cloud_cover",
        "wind_speed_forecast",
        "precipitation_forecast",
        "lng_price_index",
        "fuel_cost_unit",
        "oil_price_dubai",
        "usd_krw",
    ]
    display(model_b_final[range_cols].describe().T)

    print("negative SMP count:", int(model_b_final["is_negative_smp"].sum()))
    print("negative SMP ratio:", float(model_b_final["is_negative_smp"].mean()))

    save_path = PREP_DIR / "model_b_data.csv"
    model_b_final.to_csv(save_path, index=False, encoding="utf-8-sig")
    print("saved:", save_path)

    saved = pd.read_csv(save_path, encoding="utf-8-sig")
    saved["timestamp"] = pd.to_datetime(saved["timestamp"])
    audit_time_index(saved, name="model_b_data_saved")
    assert_final_timestamp(saved, "model_b_data_saved")

    if saved["timestamp"].duplicated().sum() != 0:
        raise ValueError("저장본 timestamp 중복이 있습니다.")

    print("model_b_data.csv 저장 및 재검수 완료")
    return saved



## 9. 전체 실행

아래 셀 하나를 실행하면 RAW 파일에서 중간 산출물 CSV와 최종 `model_b_data.csv`까지 모두 생성한다.



In [33]:
RUN_ALL = True

if RUN_ALL:
    smp_features = build_smp_features()
    calendar_features = build_calendar_features()
    weather_features = build_weather_features()
    renewable_actual = build_renewable_actual()
    renewable_forecast = build_renewable_forecast()
    fuel_cost_features = build_fuel_cost_features()
    oil_usd_features = build_oil_usd_features()
    model_b_data = merge_model_b_data()

    print("\nFINAL CHECK")
    print("model_b_data shape:", model_b_data.shape)
    print("timestamp min:", model_b_data["timestamp"].min())
    print("timestamp max:", model_b_data["timestamp"].max())
    print("missing timestamp count:", len(FINAL_INDEX.difference(pd.DatetimeIndex(model_b_data["timestamp"]))))
    print("duplicate timestamp rows:", int(model_b_data["timestamp"].duplicated().sum()))
    print("negative SMP count:", int(model_b_data["is_negative_smp"].sum()))
    display(model_b_data.head())
    display(model_b_data.tail())


제주 SMP: 제주_시간별SMP.csv cp949 (708, 28)
육지 SMP: 육지_시간별SMP.csv cp949 (708, 28)
제주 기간 중복 row 수: 0
제주 2024-05-31 row count: 1
제주 2024-05-31 24시 -> 2024-06-01 00:00:00 값: 139.63
육지 기간 중복 row 수: 0
육지 2024-05-31 row count: 1
육지 2024-05-31 24시 -> 2024-06-01 00:00:00 값: 139.63


,name,rows,columns,min_timestamp,max_timestamp,duplicate_timestamp_rows,expected_rows,unique_timestamps_in_final,missing_timestamp_count,extra_timestamp_count
0,jeju_smp_long_raw,16992,2,2024-05-31 01:00:00,2026-05-09,0,16056,16056,0,936


결측 없음
기간 외 timestamp 앞부분: [Timestamp('2024-05-31 01:00:00'), Timestamp('2024-05-31 02:00:00'), Timestamp('2024-05-31 03:00:00'), Timestamp('2024-05-31 04:00:00'), Timestamp('2024-05-31 05:00:00'), Timestamp('2024-05-31 06:00:00'), Timestamp('2024-05-31 07:00:00'), Timestamp('2024-05-31 08:00:00'), Timestamp('2024-05-31 09:00:00'), Timestamp('2024-05-31 10:00:00')]
기간 외 timestamp 뒷부분: [Timestamp('2026-05-08 15:00:00'), Timestamp('2026-05-08 16:00:00'), Timestamp('2026-05-08 17:00:00'), Timestamp('2026-05-08 18:00:00'), Timestamp('2026-05-08 19:00:00'), Timestamp('2026-05-08 20:00:00'), Timestamp('2026-05-08 21:00:00'), Timestamp('2026-05-08 22:00:00'), Timestamp('2026-05-08 23:00:00'), Timestamp('2026-05-09 00:00:00')]


,name,rows,columns,min_timestamp,max_timestamp,duplicate_timestamp_rows,expected_rows,unique_timestamps_in_final,missing_timestamp_count,extra_timestamp_count
0,mainland_smp_long_raw,16992,2,2024-05-31 01:00:00,2026-05-09,0,16056,16056,0,936


결측 없음
기간 외 timestamp 앞부분: [Timestamp('2024-05-31 01:00:00'), Timestamp('2024-05-31 02:00:00'), Timestamp('2024-05-31 03:00:00'), Timestamp('2024-05-31 04:00:00'), Timestamp('2024-05-31 05:00:00'), Timestamp('2024-05-31 06:00:00'), Timestamp('2024-05-31 07:00:00'), Timestamp('2024-05-31 08:00:00'), Timestamp('2024-05-31 09:00:00'), Timestamp('2024-05-31 10:00:00')]
기간 외 timestamp 뒷부분: [Timestamp('2026-05-08 15:00:00'), Timestamp('2026-05-08 16:00:00'), Timestamp('2026-05-08 17:00:00'), Timestamp('2026-05-08 18:00:00'), Timestamp('2026-05-08 19:00:00'), Timestamp('2026-05-08 20:00:00'), Timestamp('2026-05-08 21:00:00'), Timestamp('2026-05-08 22:00:00'), Timestamp('2026-05-08 23:00:00'), Timestamp('2026-05-09 00:00:00')]


...

,name,rows,columns,min_timestamp,max_timestamp,duplicate_timestamp_rows,expected_rows,unique_timestamps_in_final,missing_timestamp_count,extra_timestamp_count
0,model_b_data_saved,16056,36,2024-06-01,2026-03-31 23:00:00,0,16056,16056,0,0


,missing_count
renewable_pen,16056
predicted_demand,16056
net_load,16056
lag_smp_168h,168
oil_price_dubai,48
lag_smp_24h,24
smp_rolling_24h_mean,24
jeju_mainland_gap_lag_24h,24
mainland_smp_lag_24h,24
lag_smp_1h,1


model_b_data.csv 저장 및 재검수 완료

FINAL CHECK
model_b_data shape: (16056, 36)
timestamp min: 2024-06-01 00:00:00
timestamp max: 2026-03-31 23:00:00
missing timestamp count: 0
duplicate timestamp rows: 0
negative SMP count: 76


,timestamp,날짜,시간,jeju_smp,is_negative_smp,hour_sin,hour_cos,day_of_week,is_weekend,is_holiday,holiday_type,special_day_window,special_day_type,special_day_offset,peak_index,lag_smp_1h,lag_smp_24h,lag_smp_168h,smp_rolling_24h_mean,mainland_smp,mainland_smp_lag_24h,jeju_mainland_gap_lag_24h,predicted_demand,temperature_forecast,solar_radiation,cloud_cover,wind_speed_forecast,precipitation_forecast,forecast_solar,forecast_wind,renewable_pen,net_load,lng_price_index,fuel_cost_unit,oil_price_dubai,usd_krw
0,2024-06-01 00:00:00,2024-06-01,00:00:00,139.63,0,0.000000,1.000000,5,1,1,0,0,0,0,0.18,NaN,NaN,NaN,NaN,139.63,NaN,NaN,NaN,17.750,0.0,3,2.775,0.0,0.0,9.624085,NaN,NaN,100.0,130.587,NaN,1376.5
1,2024-06-01 01:00:00,2024-06-01,01:00:00,100.33,0,0.258819,0.965926,5,1,1,0,0,0,0,0.18,139.63,NaN,NaN,NaN,100.33,NaN,NaN,NaN,17.775,0.0,8,3.675,0.0,0.0,12.198507,NaN,NaN,100.0,130.587,NaN,1376.5
2,2024-06-01 02:00:00,2024-06-01,02:00:00,93.74,0,0.500000,0.866025,5,1,1,0,0,0,0,0.18,100.33,NaN,NaN,NaN,93.74,NaN,NaN,NaN,17.625,0.0,6,2.650,0.0,0.0,8.084797,NaN,NaN,100.0,130.587,NaN,1376.5
3,2024-06-01 03:00:00,2024-06-01,03:00:00,90.02,0,0.707107,0.707107,5,1,1,0,0,0,0,0.18,93.74,NaN,NaN,NaN,90.02,NaN,NaN,NaN,17.650,0.0,5,3.025,0.0,0.0,9.451972,NaN,NaN,100.0,130.587,NaN,1376.5
4,2024-06-01 04:00:00,2024-06-01,04:00:00,86.94,0,0.866025,0.500000,5,1,1,0,0,0,0,0.18,90.02,NaN,NaN,NaN,86.94,NaN,NaN,NaN,17.600,0.0,7,3.175,0.0,0.0,9.921971,NaN,NaN,100.0,130.587,NaN,1376.5


,timestamp,날짜,시간,jeju_smp,is_negative_smp,hour_sin,hour_cos,day_of_week,is_weekend,is_holiday,holiday_type,special_day_window,special_day_type,special_day_offset,peak_index,lag_smp_1h,lag_smp_24h,lag_smp_168h,smp_rolling_24h_mean,mainland_smp,mainland_smp_lag_24h,jeju_mainland_gap_lag_24h,predicted_demand,temperature_forecast,solar_radiation,cloud_cover,wind_speed_forecast,precipitation_forecast,forecast_solar,forecast_wind,renewable_pen,net_load,lng_price_index,fuel_cost_unit,oil_price_dubai,usd_krw
16051,2026-03-31 19:00:00,2026-03-31,19:00:00,127.59,0,-0.965926,0.258819,1,0,0,0,0,0,0,1.00,114.91,119.35,133.70,110.471667,127.59,119.35,0.0,NaN,13.500,0.03,10,2.875,0.0,6.053806,3.659121,NaN,NaN,90.190624,114.8859,121.1,1513.4
16052,2026-03-31 20:00:00,2026-03-31,20:00:00,127.32,0,-0.866025,0.500000,1,0,0,0,0,0,0,1.00,127.59,119.67,133.69,110.815000,127.32,119.67,0.0,NaN,13.200,0.00,10,2.200,0.0,4.009900,2.570012,NaN,NaN,90.190624,114.8859,121.1,1513.4
16053,2026-03-31 21:00:00,2026-03-31,21:00:00,123.93,0,-0.707107,0.707107,1,0,0,0,0,0,0,1.00,127.32,114.56,122.57,111.133750,121.56,114.56,0.0,NaN,13.025,0.00,10,1.850,0.0,3.280181,1.654085,NaN,NaN,90.190624,114.8859,121.1,1513.4
16054,2026-03-31 22:00:00,2026-03-31,22:00:00,123.93,0,-0.500000,0.866025,1,0,0,0,0,0,0,0.55,123.93,113.53,122.51,111.524167,121.56,113.53,0.0,NaN,13.175,0.00,10,1.800,0.0,0.000000,1.563732,NaN,NaN,90.190624,114.8859,121.1,1513.4
16055,2026-03-31 23:00:00,2026-03-31,23:00:00,120.87,0,-0.258819,0.965926,1,0,0,0,0,0,0,0.55,123.93,111.97,122.51,111.957500,120.87,111.97,0.0,NaN,13.300,0.00,10,1.300,0.0,0.000000,1.037312,NaN,NaN,90.190624,114.8859,121.1,1513.4
